# Cyber Trend Forecasting Transformer Pipeline

**Vignette Description:**
This notebook acts as the Pipeline Dashboard for the project. It handles the end-to-end workflow:
1.  **Setup:** Automatically detects if running on Google Colab or the NATO Local Server.
2.  **Training:** Executes `Train_Graph.py` to optimise the PDFormer model.
3.  **Evaluation:** Runs `Evaluate_Graph.py` to generate metrics and predictions.
4.  **Visualisation:** Runs `Visualise_Results.py` and displays the Gap Analysis inline.

## Universal Setup

This cell handles the setup logic: mounting Google Drive if on Colab, or using relative paths if on the GPU server

In [ ]:
import os
import sys

# --- CONFIGURATION ---
# If on Colab, ensure you have set the Secret 'project_path' to your Project Root
# e.g., /content/drive/MyDrive/NATO_Project_Root

try:
    # 1. CHECK ENVIRONMENT: Try to import Colab-specific modules
    import google.colab
    from google.colab import drive, userdata
    IS_COLAB = True
    print("--- Detected Environment: GOOGLE COLAB ---")
except ImportError:
    IS_COLAB = False
    print("--- Detected Environment: LOCAL SERVER (NATO/BBK) ---")

# 2. PATH SETUP
if IS_COLAB:
    # --- COLAB SETUP ---
    try:
        drive.mount('/content/drive')

        # Get the PARENT ROOT from Secrets
        PROJECT_ROOT = userdata.get('project_path')
        if not PROJECT_ROOT:
            raise ValueError("Secret 'project_path' is missing.")

    except Exception as e:
        print(f"CRITICAL COLAB SETUP ERROR: {e}")
        # Stop execution if we can't find the files
        raise e
else:
    # --- LOCAL SETUP ---
    # We assume this notebook is in 'Notebooks/', so Root is one level up
    # os.getcwd() returns /.../Project_Root/Notebooks
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# 3. CONFIGURE WORKING DIRECTORY
# We need to run scripts from inside 'Transformer_Pipeline'
PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'Transformer_Pipeline')

if not os.path.exists(PIPELINE_DIR):
    raise FileNotFoundError(f"Could not find pipeline folder at: {PIPELINE_DIR}")

# Change CWD to the pipeline folder so scripts can find config/data
os.chdir(PIPELINE_DIR)
print(f"SUCCESS: Working Directory set to: {os.getcwd()}")

# 4. ADD PROJECT ROOT TO SYS.PATH
# This allows us to import 'Notebooks.colab_utils' if needed
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added Project Root to sys.path: {PROJECT_ROOT}")

# 5. INSTALL DEPENDENCIES (COLAB ONLY)
if IS_COLAB:
    print("\n--- Installing Dependencies (Colab Only) ---")
    # We import the helper script from the sibling folder
    try:
        from Notebooks import colab_utils
        colab_utils.setup_environment()
    except ImportError:
        print("WARNING: Could not import Notebooks.colab_utils. Skipping auto-install.")

## Step 1: Train Graph Model (PDFormer)

We now execute the training script.
* **Input:** `Cyber_Trend_Graph_Config.py` (Hyperparameters)
* **Output:** `best_model_graph.pth` (Saved weights)

*Note: If running on T4 GPU, should take 15-25 minutes.*

In [ ]:
print("--- Step 1: Training PDFormer Graph Model ---")
# We use %run to execute the script in a fresh namespace.
# This runs the script as if it were in the terminal, showing live progress bars.
# This is cleaner for demo purposes than importing the main function.
%run Train_Graph.py

## Step 2: Evaluation & Metrics

This step loads the best model saved above and runs it against the Test Set.
* It calculates **RSE, RAE, and MAE** across horizons (3, 6, 12, 24 months).
* It saves the raw predictions to `.npy` files for visualisation.
* It exports the **Column Names** metadata.

In [ ]:
print("--- Step 2: Evaluating Model on Test Set ---")
%run Evaluate_Graph.py

## Step 3: Visualisation

Finally, we generate the paper-ready plots.
* **Forecast vs Actual:** Checks model accuracy on a specific node.
* **Gap Analysis:** Visualises the "Risk Gap" between **DDoS-ALL** (Threat) and **IDS/IPS Solutions** (Mitigation).

In [ ]:
print("--- Step 3: Generating Visualisations ---")

# 1. Run the visualisation script to create PNGs
%run Visualise_Results.py

# 2. Display the images inline
import os
from IPython.display import Image, display

results_dir = 'Results'
gap_plot = os.path.join(results_dir, 'gap_analysis.png')
forecast_plot = os.path.join(results_dir, 'forecast_vs_actual.png')

print("\n--- GAP ANALYSIS: Threat vs Mitigation ---")
if os.path.exists(gap_plot):
    display(Image(filename=gap_plot))
else:
    print("Gap analysis plot not found.")

print("\n--- FORECAST ACCURACY: Ground Truth vs Prediction ---")
if os.path.exists(forecast_plot):
    display(Image(filename=forecast_plot))
else:
    print("Forecast plot not found.")